# Final model: parameter count and test accuracy

Final submission: run `6e63a995e2d6` (ResNet, widths 88/128/216, 4 blocks per stage, trained on all 50,000
training images, seed 1), flip test-time augmentation. Weights and config are in `results/final_model/`.

This notebook
1. rebuilds the model and prints its parameter count (limit: 5,000,000),
2. predicts the Kaggle test set from the saved weights and checks that the predictions match the submitted file,
3. reports the test accuracy.

The Kaggle test labels are hidden, so test accuracy is the score Kaggle returned for the submitted predictions.
Step 2 needs the competition data unpacked into `data/` (`data/test.csv`, `data/test_images/*.png`); it is not
redistributed in this repository. Runs on CPU in about a minute.

In [1]:
import csv, json
from pathlib import Path

import numpy as np
import torch
from PIL import Image

from src import model as M, inference as I
from src.config import DATA_DIR, ROOT

FINAL = ROOT / "results" / "final_model"
cfg = json.loads((FINAL / "config.json").read_text())
net = M.build(cfg["model"])
net.load_state_dict(torch.load(FINAL / "model.pt", map_location="cpu"))
net.eval()
print(cfg["model"])

{'widths': [88, 128, 216], 'blocks': [4, 4, 4], 'block': 'post', 'shortcut': 'conv'}


## 1. Parameter count

In [2]:
n_params = M.count_params(net)
print(f"trainable parameters: {n_params:,}  (limit 5,000,000; {'OK' if n_params <= 5_000_000 else 'OVER'})")

trainable parameters: 4,930,530  (limit 5,000,000; OK)


## 2. Predict the Kaggle test set (flip TTA) and compare with the submitted file

In [3]:
with open(DATA_DIR / "test.csv", newline="") as f:
    rows = list(csv.DictReader(f))
ids = np.array([int(r["id"]) for r in rows])
x = np.stack([np.asarray(Image.open(DATA_DIR / "test_images" / r["filename"]).convert("RGB")) for r in rows])
print("test images:", x.shape)

probs = I.predict_views(net, x, "cpu", batch=250, views=I.TTA["flip"])   # identity + mirror
pred = probs.astype(np.float32).mean(1).argmax(1)

with open(ROOT / "submissions" / "sub10_long_screen_s1_M3E_R2_foldfinal_seed1_flip.csv", newline="") as f:
    sub = {int(r["id"]): int(r["label"]) for r in csv.DictReader(f)}
submitted = np.array([sub[i] for i in ids])
agree = (pred == submitted).mean()
print(f"agreement with submitted predictions: {agree:.2%} ({(pred != submitted).sum()} of {len(ids)} differ)")

test images: (2000, 32, 32, 3)


agreement with submitted predictions: 99.90% (2 of 2000 differ)


The submitted predictions were made on a GPU in bfloat16; this notebook runs in float32 on the CPU, so a
handful of borderline images can flip class. Agreement near 100% confirms these are the submitted weights.

## 3. Test accuracy

In [4]:
KAGGLE = {"public": 0.913, "private": 0.889}   # scores returned by Kaggle for sub10 (labels are hidden)
print(f"parameters:            {n_params:,}")
print(f"test accuracy (public leaderboard):  {KAGGLE['public']:.1%}")
print(f"test accuracy (private leaderboard): {KAGGLE['private']:.1%}  <- final score")

parameters:            4,930,530
test accuracy (public leaderboard):  91.3%
test accuracy (private leaderboard): 88.9%  <- final score
